# Calculate Depression Depths (mm) per Subbasin

In [1]:
import os
import sys
import fileinput
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint

sys.path.append("..")
from src.py.proj_conf import *

## Import input files

In [ ]:
# CLRH basin delineation shapefile
mc_clrh_gdf = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/CLRH_RoutingProduct/05OE006')) + "/" + "finalcat_info_v1-0.shp")

In [93]:
# Project clrh basins and write to shapefile
mc_clrh_gdf_projected = mc_clrh_gdf.to_crs(4617)

mc_clrh_gdf_projected.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "clrh_basins_projected.shp")

## Import WhiteboxTools

In [5]:
from WBT.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()

In [21]:
os.getcwd()

'c:\\Users\\emilito\\Documents\\PWA\\_notebooks\\WBT'

In [41]:
wbt.set_working_dir(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')))

In [42]:
wbt.get_working_dir()

'c:\\Users\\emilito\\Documents\\PWA\\Data\\Raw\\manning\\WB_Data'

## Process DEM and Stream Network

In [98]:
######### SUBBASIN AND STREAM SHAPEFILES WERE PROJECTED IN ARCGIS PRO (NO PROJECTION TOOL IN WHITEBOX)

# Clip NHN streams to study area
wbt.clip(
    i = 'NHN_05OE000_5_0_HD_SLWATER_1.shp',
    clip = 'clrh_basins_projected.shp',
    output = 'NHN_Clip_wbt.shp'
)

######### RASTER DATA WAS CLIPPED AND RESAMPLED IN ARCGIS PRO DUE TO MEMORY ALLOCATION ERROR

# Clip dem raster to study area
#wbt.clip_raster_to_polygon(
    #i = 'sr_dem_cgvd28.tif',
    #polygons = 'finalcat_info_v1-0_Project.shp',
    #output = 'sr_dem_cgvd28_Clip_MC.tif'
#)

# Resample DEM to 5m (for faster processing)
#wbt.resample(
    #inputs = 'sr_dem_cgvd28_Clip_MC.tif',
    #output = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr',
    #cell_size = 5,
    #method = "cc"
#)

######### BELOW CODE WORKS BUT IS NOT CURRENTLY NECESSARY BECAUSE STREAMS ARE RASTERIZED WITHIN FILL BURN TOOL

# Convert stream network to raster
#wbt.vector_lines_to_raster(
    #i = 'NHN_Clip_wbt_Project.shp',
    #output = 'NHN_Clip_Project_raster_wbt.tif',
    #field = "FID",
    #nodata = True,
    #cell_size = 5.0
#)

.\whitebox_tools.exe --run="Clip" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --input='NHN_05OE000_5_0_HD_SLWATER_1.shp' --clip='clrh_basins_projected.shp' --output='NHN_Clip_wbt.shp' -v --compress_rasters=False

****************************
* Welcome to Clip          *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progress: 31%
Progress: 32%
Progress: 33%
Progress: 34%
Progress: 35%
Progress: 36%
Progress: 37%
Progress: 38%
Progress: 39%
Progress: 40%
Progress: 41%
Progress: 42%
Progress: 43%
P

0

In [99]:
# Project NHN clipped shapefile
mc_streams_clip = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip_wbt.shp")
mc_streams_clip_proj = mc_streams_clip.to_crs(26914)
mc_streams_clip_proj.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip_Project.shp")

## Generate Depressions Raster

In [100]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif',
    streams = 'NHN_Clip_Project.shp',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn.tif'
)

# Subtract raw DEM from filled DEM
wbt.raster_calculator(
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    statement = "'DEM_Clip_MC_Resample_5m_FillBurn.tif' - 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif'"
)

# Remove stray burn lines
wbt.conditional_evaluation(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    statement = "value < 0.0",
    true = 0.0,
    false = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif'
)

.\whitebox_tools.exe --run="FillBurn" --wd="c:\Users\emilito\Documents\PWA\Data\Raw\manning\WB_Data" --dem='sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif' --streams='NHN_Clip_Project.shp' --output='DEM_Clip_MC_Resample_5m_FillBurn.tif' -v --compress_rasters=False

****************************
* Welcome to FillBurn      *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading streams data...
Reading DEM data...
Rasterizing Streams: 0%
Rasterizing Streams: 1%
Rasterizing Streams: 2%
Rasterizing Streams: 3%
Rasterizing Streams: 4%
Rasterizing Streams: 5%
Rasterizing Streams: 6%
Rasterizing Streams: 7%
Rasterizing Streams: 8%
Rasterizing Streams: 9%
Rasterizing Streams: 10%
Rasterizing Streams: 11%
Rasterizing Streams: 12%
Rasterizing Streams: 13%
Rasterizing Streams: 14%
Rasterizing Streams: 15%
Rasterizing Streams: 16%
Rasterizing Streams: 17%
Rasterizing Streams: 18%
Rasterizing Streams: 19%
Rasterizing Streams: 20%
Rasterizing Streams: 21%
Rasterizi

0